In [31]:
import pandas as pd
from PIL import Image
import requests
import os
import io

In [2]:
df = pd.read_csv("./data/uniqlo_processed.csv")

In [6]:
cc = {}
val = []
for item in df.iloc:
    val.append(cc.get(item.id, 0))
    cc[item.id] = cc.get(item.id, 0) + 1
    

In [8]:
df["local_id"] = val

In [9]:
df.head()

,id,price,item_url,Item_name,Item_type,materials,img_url,is_good,local_id
0,469869,79.9,https://www.uniqlo.com/us/en/products/E469869-...,Ultra Light Down Jacket,WOMEN/outerwear-and-blazers/ultra-light-down,Face: 100% Nylon/ Filling: Down ( Minimum 90% ...,https://image.uniqlo.com/UQ/ST3/us/imagesgoods...,True,0
1,469869,79.9,https://www.uniqlo.com/us/en/products/E469869-...,Ultra Light Down Jacket,WOMEN/outerwear-and-blazers/ultra-light-down,Face: 100% Nylon/ Filling: Down ( Minimum 90% ...,https://image.uniqlo.com/UQ/ST3/us/imagesgoods...,True,1
2,469869,79.9,https://www.uniqlo.com/us/en/products/E469869-...,Ultra Light Down Jacket,WOMEN/outerwear-and-blazers/ultra-light-down,Face: 100% Nylon/ Filling: Down ( Minimum 90% ...,https://image.uniqlo.com/UQ/ST3/us/imagesgoods...,True,2
3,469869,79.9,https://www.uniqlo.com/us/en/products/E469869-...,Ultra Light Down Jacket,WOMEN/outerwear-and-blazers/ultra-light-down,Face: 100% Nylon/ Filling: Down ( Minimum 90% ...,https://image.uniqlo.com/UQ/ST3/us/imagesgoods...,True,3
4,469869,79.9,https://www.uniqlo.com/us/en/products/E469869-...,Ultra Light Down Jacket,WOMEN/outerwear-and-blazers/ultra-light-down,Face: 100% Nylon/ Filling: Down ( Minimum 90% ...,https://image.uniqlo.com/UQ/ST3/us/imagesgoods...,True,4


In [42]:
df.to_csv("./data/uniqlo_processed.csv")

## GCS utils

In [18]:
from google.cloud import storage
import os

def upload_to_gcp(bucket_name, source_file_path, destination_blob_name):
    """
    Uploads a file to a Google Cloud Storage bucket.

    Args:
        bucket_name (str): Name of the GCP bucket.
        source_file_path (str): Local path to the file to be uploaded.
        destination_blob_name (str): Name of the file in the GCP bucket.
    """
    try:
        # Initialize a GCP client
        storage_client = storage.Client()

        # Get the bucket
        bucket = storage_client.bucket(bucket_name)

        # Create a blob object from the bucket
        blob = bucket.blob(destination_blob_name)

        # Upload the file to GCP
        blob.upload_from_filename(source_file_path)

        print(f"File {source_file_path} uploaded to {destination_blob_name} in bucket {bucket_name}.")
    except Exception as e:
        print(f"An error occurred: {e}")

## Uploading to GCS

In [14]:
# df.iloc[0].img_url
img = Image.open(requests.get(df.iloc[0].img_url, stream=True).raw)

In [ ]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/home/abhisheksureddy/sharp-messenger-444918-a4-538e8b5aecdb.json"
bucket_name = "cs-data-bucket-1"
# Initialize a GCP client
storage_client = storage.Client()

# Get the bucket
bucket = storage_client.bucket(bucket_name)
blob = bucket.blob("sample_image.png")
# img.save("sample_image.png")
blob.upload_from_filename("sample_image.png")

In [ ]:
source_file_path = "/home/jupyter/repos/fashion-retrieval/features-cache/uniqlo_vit32.pkl"
destination_blob_name = "features-cache/uniqlo_vit32.pkl"

# Upload the file
upload_to_gcp(bucket_name, source_file_path, destination_blob_name)


In [24]:
# read back the image
i_bytes = blob.download_as_bytes()
img2 = Image.open(io.BytesIO(i_bytes))

## Save all uniqlo images in GCS

In [37]:
from tqdm import tqdm

In [39]:
failed_objs = []
for item in tqdm(df.iloc):
    try:
        uniq_id = f"{item.id}_{item.local_id}"
        img = Image.open(requests.get(item.img_url, stream=True).raw)
        img.save("sample_image.png")
        destination_blob_name = f"uniqlo_data/{uniq_id}.png"
        blob = bucket.blob(destination_blob_name)
        blob.upload_from_filename("sample_image.png")
    except:
        failed_objs.append(uniq_id)


5329it [55:01,  1.61it/s]


In [40]:
len(failed_objs)

233

In [41]:
5329 - 233

5096

In [6]:
import threading

num_threads = threading.active_count()

print("Number of active threads:", num_threads)

Number of active threads: 7


In [1]:
!lscpu

Architecture:        x86_64
CPU op-mode(s):      32-bit, 64-bit
Byte Order:          Little Endian
Address sizes:       46 bits physical, 48 bits virtual
CPU(s):              16
On-line CPU(s) list: 0-15
Thread(s) per core:  2
Core(s) per socket:  8
Socket(s):           1
NUMA node(s):        1
Vendor ID:           GenuineIntel
CPU family:          6
Model:               85
Model name:          Intel(R) Xeon(R) CPU @ 2.80GHz
Stepping:            7
CPU MHz:             2800.226
BogoMIPS:            5600.45
Hypervisor vendor:   KVM
Virtualization type: full
L1d cache:           32K
L1i cache:           32K
L2 cache:            1024K
L3 cache:            33792K
NUMA node0 CPU(s):   0-15
Flags:               fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonstop_tsc cpuid tsc_known_freq pni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2apic movbe popcnt aes xsave avx f

In [10]:
!gsutil ls gs://cs-data-bucket-1/fashion200k/** | wc -l

201550
